# 08_build_evaluation_data

固定した元メタデータから無加工正例データを生成し、それを元に誤植付き正例データと負例データを作成して固定ファイルへ保存する。

## 負例タイトルの AI 書換えタイミング

負例データの論文タイトルは、この notebook の前半ではまだ AI で書き換えない。

1. まず `negative_examples.json` の土台を作る
2. 次に `negative_title_rewrite_requests.json` を書き出す
3. `source_id` ごとに 1 回だけ AI へタイトル書換えを依頼する
4. 得られた書換え後タイトルを `negative_title_rewrite_results.json` に保存する
5. その結果を使って、同じ `source_id` の 3 スタイルへ同じ内容を反映する

つまり、AI への依頼は負例データの土台と依頼一覧を保存した後に行う。


In [ ]:
# Notebook から src 配下の自作モジュールを import できるようにする設定
from pathlib import Path
import sys

project_root = Path.cwd().parent
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))


In [ ]:
# 設定値と評価データ生成関数を読み込む
from pprint import pprint

from tanigawa_shoshi.config import (
    BASE_POSITIVE_EXAMPLES_PATH,
    NEGATIVE_EXAMPLES_PATH,
    NEGATIVE_TITLE_REWRITE_REQUESTS_PATH,
    NEGATIVE_TITLE_REWRITE_RESULTS_PATH,
    POSITIVE_EXAMPLES_PATH,
    SAMPLED_SOURCE_DOCS_PATH,
)
from tanigawa_shoshi.evaluation_data import (
    apply_negative_title_rewrite_results,
    build_base_positive_examples,
    build_negative_examples,
    build_negative_title_rewrite_requests,
    build_negative_title_rewrite_results,
    build_positive_examples,
    load_examples,
    load_source_docs,
    save_examples,
)


In [ ]:
# 元メタデータを読み込む
source_docs = load_source_docs()

print(f"source docs path: {SAMPLED_SOURCE_DOCS_PATH}")
print(f"loaded source docs: {len(source_docs)}")
print(f"base positive output path: {BASE_POSITIVE_EXAMPLES_PATH}")
print(f"positive output path: {POSITIVE_EXAMPLES_PATH}")
print(f"negative output path: {NEGATIVE_EXAMPLES_PATH}")
print(f"negative title rewrite request path: {NEGATIVE_TITLE_REWRITE_REQUESTS_PATH}")
print(f"negative title rewrite result path: {NEGATIVE_TITLE_REWRITE_RESULTS_PATH}")


In [ ]:
# 無加工正例データを生成する
base_positive_result = build_base_positive_examples(source_docs)

print("base positive build stats")
pprint(base_positive_result["stats"])
print()
print("style names")
print(base_positive_result["style_names"])
print()
print(f"built examples: {len(base_positive_result['examples'])}")


In [ ]:
# 無加工正例データを固定ファイルへ保存する
saved_base_path = save_examples(
    BASE_POSITIVE_EXAMPLES_PATH,
    base_positive_result["examples"],
)
print(saved_base_path)


In [ ]:
# 保存済み無加工正例データを再読込し、先頭数件を確認する
base_positive_examples = load_examples(BASE_POSITIVE_EXAMPLES_PATH)

print(f"loaded examples: {len(base_positive_examples)}")
print()

for index, example in enumerate(base_positive_examples[:3], start=1):
    print("=" * 60)
    print(f"index: {index}")
    print(f"example_id: {example.get('example_id')}")
    print(f"style: {example.get('style')}")
    print(f"doi: {example.get('doi')}")
    print(f"field_names: {example.get('field_names')}")
    print(f"reference_fields: {example.get('reference_fields')}")
    print(f"reference_text: {example.get('reference_text')}")


In [ ]:
# 無加工正例データから誤植付き正例データを生成する
positive_seed = 42
positive_result = build_positive_examples(
    base_positive_examples,
    seed=positive_seed,
)

print("positive build stats")
pprint(positive_result["stats"])
print()
print(f"positive seed: {positive_seed}")
print(f"built examples: {len(positive_result['examples'])}")


In [ ]:
# 誤植付き正例データを固定ファイルへ保存する
saved_positive_path = save_examples(
    POSITIVE_EXAMPLES_PATH,
    positive_result["examples"],
)
print(saved_positive_path)


In [ ]:
# 保存済み誤植付き正例データを再読込し、誤植内容を確認する
positive_examples = load_examples(POSITIVE_EXAMPLES_PATH)
typo_examples = [example for example in positive_examples if example.get("has_typos")]

print(f"loaded examples: {len(positive_examples)}")
print(f"typo examples: {len(typo_examples)}")
print()

for index, example in enumerate(typo_examples[:3], start=1):
    print("=" * 60)
    print(f"index: {index}")
    print(f"example_id: {example.get('example_id')}")
    print(f"style: {example.get('style')}")
    print(f"typo_details: {example.get('typo_details')}")
    print(f"reference_fields: {example.get('reference_fields')}")
    print(f"reference_text: {example.get('reference_text')}")


In [ ]:
# 無加工正例データから負例データの土台を生成する
negative_result = build_negative_examples(base_positive_examples)

print("negative build stats")
pprint(negative_result["stats"])
print()
print(f"built examples: {len(negative_result['examples'])}")


In [ ]:
# 負例データの土台を固定ファイルへ保存する
saved_negative_path = save_examples(
    NEGATIVE_EXAMPLES_PATH,
    negative_result["examples"],
)
print(saved_negative_path)


In [ ]:
# タイトル AI 書換え依頼用の一覧を source_id 単位で作成する
negative_examples = load_examples(NEGATIVE_EXAMPLES_PATH)
rewrite_request_result = build_negative_title_rewrite_requests(negative_examples)

print("title rewrite request stats")
pprint(rewrite_request_result["stats"])
print()
print(f"request count: {len(rewrite_request_result['requests'])}")


In [ ]:
# タイトル AI 書換え依頼用の一覧を固定ファイルへ保存する
saved_request_path = save_examples(
    NEGATIVE_TITLE_REWRITE_REQUESTS_PATH,
    rewrite_request_result["requests"],
)
print(saved_request_path)


In [ ]:
# 保存済み負例データの土台を再読込し、先頭数件を確認する
negative_examples = load_examples(NEGATIVE_EXAMPLES_PATH)

print(f"loaded examples: {len(negative_examples)}")
print()

for index, example in enumerate(negative_examples[:3], start=1):
    print("=" * 60)
    print(f"index: {index}")
    print(f"example_id: {example.get('example_id')}")
    print(f"style: {example.get('style')}")
    print(f"field_names: {example.get('field_names')}")
    print(f"needs_title_rewrite: {example.get('needs_title_rewrite')}")
    print(f"title_rewrite_status: {example.get('title_rewrite_status')}")
    print(f"negative_details: {example.get('negative_details')}")
    print(f"reference_fields: {example.get('reference_fields')}")
    print(f"reference_text: {example.get('reference_text')}")


In [ ]:
# 保存済みタイトル AI 書換え依頼一覧を再読込し、先頭数件を確認する
rewrite_requests = load_examples(NEGATIVE_TITLE_REWRITE_REQUESTS_PATH)

print(f"loaded requests: {len(rewrite_requests)}")
print()

for index, request in enumerate(rewrite_requests[:3], start=1):
    print("=" * 60)
    print(f"index: {index}")
    print(f"source_id: {request.get('source_id')}")
    print(f"original_title: {request.get('original_title')}")
    print(f"style_names: {request.get('style_names')}")
    print(f"prompt: {request.get('prompt')}")


## AI 書換え結果の保存と反映

`negative_title_rewrite_requests.json` を元に `source_id` ごとの書換え後タイトルを作成したら、
`negative_title_rewrite_results.json` として保存する。

この後のセルは、その保存済み結果を読み込んで `negative_examples.json` に反映する。


In [ ]:
# 保存済みタイトル書換え結果を再読込し、先頭数件を確認する
rewrite_results = load_examples(NEGATIVE_TITLE_REWRITE_RESULTS_PATH)

print(f"loaded rewrite results: {len(rewrite_results)}")
print()

for index, result in enumerate(rewrite_results[:3], start=1):
    print("=" * 60)
    print(f"index: {index}")
    print(f"source_id: {result.get('source_id')}")
    print(f"original_title: {result.get('original_title')}")
    print(f"rewritten_title: {result.get('rewritten_title')}")
    print(f"title_rewrite_status: {result.get('title_rewrite_status')}")


In [ ]:
# タイトル書換え結果を負例データへ反映し、再保存する
negative_examples = load_examples(NEGATIVE_EXAMPLES_PATH)
applied_negative_result = apply_negative_title_rewrite_results(
    negative_examples,
    rewrite_results,
)

print("applied negative stats")
pprint(applied_negative_result["stats"])
print()

saved_negative_path = save_examples(
    NEGATIVE_EXAMPLES_PATH,
    applied_negative_result["examples"],
)
print(saved_negative_path)


In [ ]:
# タイトル書換え反映後の負例データを再読込し、先頭数件を確認する
negative_examples = load_examples(NEGATIVE_EXAMPLES_PATH)

print(f"loaded examples: {len(negative_examples)}")
print()

for index, example in enumerate(negative_examples[:3], start=1):
    print("=" * 60)
    print(f"index: {index}")
    print(f"example_id: {example.get('example_id')}")
    print(f"style: {example.get('style')}")
    print(f"needs_title_rewrite: {example.get('needs_title_rewrite')}")
    print(f"title_rewrite_status: {example.get('title_rewrite_status')}")
    print(f"title_rewrite_result: {example.get('title_rewrite_result')}")
    print(f"reference_fields: {example.get('reference_fields')}")
    print(f"reference_text: {example.get('reference_text')}")
